# FontSense reproducible demo

This notebook is the primary assessment demo. It loads a saved model, accepts raw or unseen input, validates it, and returns probabilities without relying on hidden notebook state.

# Setup

Run this notebook from the repository root. In Google Colab, clone the GitHub repository first and replace the placeholder URL.

In [ ]:
from pathlib import Path
import os, sys

REPO_URL = "PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE"
if 'google.colab' in sys.modules:
    if not Path('/content/fontsense-capstone').exists():
        if 'PASTE_' in REPO_URL:
            raise ValueError('Replace REPO_URL with your GitHub repository URL first.')
        !git clone {REPO_URL} /content/fontsense-capstone
    os.chdir('/content/fontsense-capstone')
    %pip install -q -r requirements.txt
    %pip install -q -e .
else:
    root = Path.cwd()
    if root.name == 'notebooks':
        root = root.parent
    os.chdir(root)
    os.environ['PYTHONPATH'] = str(root / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
    if str(root / 'src') not in sys.path:
        sys.path.insert(0, str(root / 'src'))
print('Project root:', Path.cwd())


In [ ]:
from pathlib import Path
from PIL import Image
from fontsense.inference import FontSensePredictor

FINAL_ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR = FINAL_ARTIFACT_DIR if (FINAL_ARTIFACT_DIR/'hog_pipeline.joblib').exists() or (FINAL_ARTIFACT_DIR/'cnn_model.pt').exists() else FINAL_ARTIFACT_DIR/'proof'
MODEL = 'hog' if (ARTIFACT_DIR/'hog_pipeline.joblib').exists() else 'cnn'
predictor = FontSensePredictor(ARTIFACT_DIR, model=MODEL, threshold=0.55)
print('Loaded:', MODEL)

## Predict one image

In [ ]:
IMAGE_PATH = 'data/sample/serif__example.png'  # replace with an unseen crop
if not Path(IMAGE_PATH).exists():
    candidates = list(Path('data/sample').glob('*.png'))
    if not candidates:
        raise FileNotFoundError('Add an image to data/sample or run scripts/create_demo_examples.py.')
    IMAGE_PATH = str(candidates[0])
image = Image.open(IMAGE_PATH).convert('RGB')
display(image)
result = predictor.predict(image)
result

## Optional upload widget in Colab

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    uploaded_image = Image.open(filename).convert('RGB')
    display(uploaded_image)
    print(predictor.predict(uploaded_image))

## Invalid-input behavior

In [ ]:
try:
    tiny = Image.new('RGB', (5,5), 'white')
    predictor.predict(tiny)
except Exception as exc:
    print('Handled correctly:', exc)